# ConvNeXt-Tiny: 질감 선별 × 베드 지도 마스크 — 3시드 × 5fold

## 왜 돌리나
DINOv2 격자에서 이 조합은 공통 사진 3,039장 중 23장 틀려, 재활용×재활용과 공동 1등이었다 [확인: 빠른 모델 1회].
ConvNeXt에서는 아직 안 돌렸다. 이걸 채우면 선별 3종 × 마스크 비교가 아래처럼 맞물린다.

| 비교 | 같은 것 | 다른 것 |
|---|---|---|
| 질감×질감(10/05) vs **질감×베드 지도(이번)** | 선별 (질감) | 마스크 |
| 2단계×베드 지도(10/13) vs **질감×베드 지도(이번)** | 마스크 (베드 지도) | 선별 |

- 질감 선별 = 질감 게이트 비율 ≥ 10% (3,449장, 격자와 같은 기준). 옛 질감×질감은 여기서 마스크 면적 10% 이하 13장을 더 뺀 3,436장이라 학습 장수가 fold당 몇 장 다르다.

## 고정한 것
10/13 2단계 실험과 같다: 같은 엔진, 512 입력, 배치 16, 20라운드, 같은 학습률·증강, fold별 업데이트 횟수 2840/2460/2260/2580/2740, 같은 folds.csv, 평가 4,000장 전부.

## 10/13 코드와 다른 것
- 학습 선별: 2단계 → 질감 게이트. 조건 1개(v6_tex_bedmap).
- 캐시 확인 버그 수정 (회색값 튜플/리스트). 그래서 캐시를 매번 다시 만들 필요가 없다.

## 한계
클래스별 출력 1회, 독립 테스트 아님, 질감 게이트 판정 영역(250×250)과 모델 입력(고정 크롭)이 다름, 회색 면적 지름길 미해결.

## 1. 설정
Drive `산공종설/`에 아래 5개가 있어야 한다. **새로 올릴 건 `convnext_texsel_bedmap_patch_20261014.zip` 하나뿐**이다.

| 파일 | 내용 | 상태 |
|---|---|---|
| underfill_originals_4000.zip | 원본 4,000장 | 이미 있음 |
| resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip | 공통 분류 함수, folds.csv, 예산 기준 목록 | 이미 있음 |
| convnext_three_preproc_patch_20261005.zip | 질감 마스크 4,000장 | 이미 있음 |
| dinov2_grid_bundle_20261013.zip | 2단계 v3.2 목록, 베드 지도 마스크 4,000장 | 이미 있음 |
| convnext_texsel_bedmap_patch_20261014.zip | 이번 어댑터 + 엔진 복사본 | **새로 올림** |

런타임은 L4 GPU. 끊겨도 같은 RUN_NAME으로 위부터 다시 실행하면 이어서 한다 (라운드마다 저장).

In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP_NAME = 'underfill_originals_4000.zip'
BUNDLE_ZIP_NAME = 'resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip'
TEXTURE_ZIP_NAME = 'convnext_three_preproc_patch_20261005.zip'
GRID_ZIP_NAME = 'dinov2_grid_bundle_20261013.zip'
PATCH_ZIP_NAME = 'convnext_texsel_bedmap_patch_20261014.zip'
RUN_NAME = 'convnext_texsel_bedmap_20261014_run01'
IMG_SIZE = 512
BATCH = 16
ROUNDS = 20
SEEDS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]
RUN_TRAINING = True         # False면 입력 준비·점검만
RUN_REMAINING_SEEDS = True  # 시드0만 보고 멈추려면 False

In [ ]:
import torch, torchvision
from google.colab import drive
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name())
drive.mount('/content/drive')
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. 파일 풀기·무결성 점검
ZIP을 런타임 SSD(`/content`)에 푼다. Drive에서 바로 읽으면 느리다.
`--verify-only`는 학습 없이 5가지(1조건×5fold) 분할이 buffer 30프레임을 지키는지, 업데이트 횟수가 옛 실험과 같은지만 확인한다.
- 확인할 것: `eligibility_audit.csv`의 `total_update_steps`가 fold0~4에서 2840/2460/2260/2580/2740 (옛 실험과 동일)

In [ ]:
import zipfile, hashlib, json, shutil, subprocess, sys, os
import pandas as pd
from IPython.display import display
LOCAL = Path('/content/convnext_texsel_bedmap_20261014')
LOCAL.mkdir(parents=True, exist_ok=True)
BUNDLE, ORIGINALS, TEXPATCH, GRID, PATCH = (LOCAL/n for n in ('bundle', 'originals', 'texpatch', 'grid', 'patch'))
CACHE = LOCAL / f'cache_{IMG_SIZE}'

def extract_zip(source, target):
    assert source.is_file(), f'파일 없음: {source}'
    local_zip = LOCAL / source.name
    if not local_zip.exists() or local_zip.stat().st_size != source.stat().st_size:
        shutil.copy2(source, local_zip)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for item in archive.infolist():
            assert (target/item.filename).resolve().is_relative_to(target.resolve()), 'ZIP 경로 오류'
        archive.extractall(target)
    print('준비:', source.name)

for name, target in ((BUNDLE_ZIP_NAME, BUNDLE), (TEXTURE_ZIP_NAME, TEXPATCH), (GRID_ZIP_NAME, GRID),
                     (PATCH_ZIP_NAME, PATCH), (ORIGINAL_ZIP_NAME, ORIGINALS)):
    extract_zip(DRIVE_DIR / name, target)
for folder in (BUNDLE, TEXPATCH, PATCH):
    hashes = json.loads((folder/'checksums.json').read_text())
    for name, expected in hashes.items():
        assert hashlib.sha256((folder/name).read_bytes()).hexdigest() == expected, name
    print('해시 일치:', folder.name, len(hashes), '개')
os.environ['PYTHONPATH'] = str(PATCH) + os.pathsep + str(BUNDLE) + os.pathsep + os.environ.get('PYTHONPATH', '')
BASE_COMMAND = [sys.executable, '-u', str(PATCH/'train_convnext_texsel_bedmap.py'),
    '--bundle', str(BUNDLE), '--texture', str(TEXPATCH/'texture'), '--grid', str(GRID),
    '--originals', str(ORIGINALS), '--cache', str(CACHE), '--output', str(OUTPUT),
    '--size', str(IMG_SIZE), '--batch', str(BATCH), '--rounds', str(ROUNDS)]
subprocess.run(BASE_COMMAND + ['--verify-only'], check=True)
audit = pd.read_csv(OUTPUT/'eligibility_audit.csv')
display(audit)
assert audit.groupby('fold').total_update_steps.first().tolist() == [2840, 2460, 2260, 2580, 2740]
display(pd.read_csv(OUTPUT/'coverage.csv'))

## 3. 입력 캐시 만들기 + 눈으로 확인
4,000장 × 베드 지도 1벌을 512×512로 줄여 SSD에 저장한다 (약 3GB, 10분 안팎).
- 왜 미리 만드나: 20라운드 동안 같은 사진을 수십 번 읽는데, 매번 JPG를 풀고 마스크를 칠하면 GPU가 놀게 된다.
- 마스크 밖은 회색 RGB(124,116,104)로 칠한다. 기존 실험과 같은 색이다.
- 아래 그림에서 확인할 것: 베드가 보이는 사진(아래 두 줄)에서 베드 부분이 회색으로 가려졌는지.

In [ ]:
# 캐시 표시 파일(complete.json)이 이미 있으면 다시 만들지 않는다 (아래 6번 설명 참고)
if not (CACHE/'complete.json').exists():
    subprocess.run(BASE_COMMAND + ['--prepare-only'], check=True)
import numpy as np
import matplotlib.pyplot as plt
inputs = pd.read_csv(OUTPUT/'input_manifest.csv')
arrays = {n: np.load(CACHE/f'{n}.npy', mmap_mode='r') for n in ('bedmap',)}
picks = []
for clean in (1, 0):
    for label in ('normal', 'underfill_50'):
        g = inputs[(inputs.common == 1) & (inputs.clean_old == clean) & (inputs.label == label)]
        picks.append((g.index[len(g)//2], 'clean' if clean else 'bed visible'))
fig, axes = plt.subplots(len(picks), 1, figsize=(4, 4*len(picks)), squeeze=False)
for i, (idx, kind) in enumerate(picks):
    r = inputs.iloc[idx]
    for j, n in enumerate(('bedmap',)):
        axes[i, j].imshow(arrays[n][idx].transpose(1, 2, 0)); axes[i, j].axis('off')
        axes[i, j].set_title(f'{n}: {r.label} #{r.frame_index} ({kind})')
fig.tight_layout(); plt.show(); plt.close(fig)

## 4. 실행·결과 표시 함수
긴 로그는 Drive의 `train_seed*.log`에 저장하고 화면에는 fold 완료 줄만 띄운다.
결과 표에서 볼 것 (순위 비교는 `common` 3,039장에서만):
- `common`: 세 선별 모두 통과한 공통 사진. DINOv2 격자 A와 같은 사진이다.
- `common_clean` / `common_bed_visible`: 공통 사진을 깨끗한 사진 2,648 / 베드가 보이는 사진 391로 나눔. **마스크 차이는 베드가 보이는 쪽에서 나야 한다** [추정: 격자 결과 기반].
- `own_texture`: 질감 게이트가 판정한 3,449장 (운영 성능). `own_stage2`는 2단계 조건과 같은 사진으로 보는 참고값.
- `none_pass`: 세 선별 모두 보류한 539장. 정답률이 낮아도 정상이다 (판정 불가 사진이라 선별이 필요한 근거).

In [ ]:
def export_results(suffix):
    destination = DRIVE_DIR / f'{RUN_NAME}_{suffix}_results.zip'
    temp = LOCAL / destination.name
    with zipfile.ZipFile(temp, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix.lower() in {'.csv', '.json', '.log', '.png', '.md'}:
                archive.write(path, path.relative_to(OUTPUT).as_posix())
    shutil.copy2(temp, destination)
    print('공유할 결과 ZIP:', destination)
    return destination

def show_results(seed):
    progress = json.loads((OUTPUT/'progress.json').read_text())
    assert seed in progress['fully_completed_seeds'], '이 시드의 5회가 아직 완료되지 않았습니다.'
    print(f'시드 {seed} 완료 — 전체 {progress["completed_runs"]}/15회')
    s = pd.read_csv(OUTPUT/'summary.csv')
    display(s[s.seed == seed][['version', 'group', 'n', 'accuracy', 'underfill_recall', 'normal_recall', 'auroc', 'fp', 'fn']])
    m = pd.read_csv(OUTPUT/'across_seeds.csv')
    display(m[(m.group == 'common') & m.metric.isin(['accuracy', 'fp', 'fn', 'auroc'])])
    return export_results(f'seed{seed}')

def train_seed(seed):
    progress = OUTPUT/'progress.json'
    if progress.exists() and seed in json.loads(progress.read_text())['fully_completed_seeds']:
        print(f'시드 {seed}는 이미 5회 완료 — 학습 없이 결과만 다시 표시합니다.')
        return show_results(seed)
    log_path = OUTPUT / f'train_seed{seed}.log'
    command = BASE_COMMAND + ['--seeds', str(seed), '--folds', *map(str, FOLDS)]
    print(f'시드 {seed}: 1조건 × 5fold = 5회 시작. 로그: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as logfile:
        logfile.write('\n실행 명령: '+json.dumps(command, ensure_ascii=False)+'\n')
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            logfile.write(line); logfile.flush()
            if f'round {ROUNDS}/{ROUNDS}:' in line or '이미 완료' in line:
                print(line.rstrip(), flush=True)
        return_code = process.wait()
    if return_code:
        print(log_path.read_text(encoding='utf-8')[-7000:])
        raise RuntimeError(f'학습 오류. 기존 저장은 유지됩니다. 로그: {log_path}')
    return show_results(seed)

## 5. 시드 0 먼저 (약 1시간, L4 기준)
옛 실험은 L4에서 fold당 약 12분이었다 [확인: 옛 결과 complete.json]. 이번엔 5회 → 약 1시간 + 캐시 시간.
끝나면 표와 `..._seed0_results.zip`이 나온다. 이것만 먼저 공유해도 된다.

In [ ]:
assert SEEDS == [0, 1, 2] and FOLDS == [0, 1, 2, 3, 4]
if RUN_TRAINING:
    SEED0_ZIP = train_seed(0)
else:
    print('입력 준비만 완료했습니다. 학습하지 않았습니다.')

## 6. 시드 1·2 (약 2시간)
시드마다 ZIP을 따로 남기므로 중간에 끊겨도 앞 결과는 보존된다. 실험 중 설정·묶음을 바꾸지 말 것.
- 왜 3시드: ConvNeXt는 초기값·사진 순서에 따라 결과가 흔들린다. 1시드 차이가 시드 간 흔들림보다 작으면 "차이 있다"고 말할 수 없다.

In [ ]:
if RUN_TRAINING and RUN_REMAINING_SEEDS:
    for seed in SEEDS[1:]:
        train_seed(seed)
    print('3시드 × 5fold = 15회 완료')
    FINAL_ZIP = export_results('all3seeds')
elif RUN_TRAINING:
    print('시드0 결과만 보관하고 멈췄습니다.')

## 7. 공유
`..._seed0_results.zip`(중간), `..._all3seeds_results.zip`(최종)을 받아 오면 된다. 모델 가중치(약 1.5GB)는 Drive 결과 폴더에만 남는다.
기존 4개 조합 3시드 결과와의 비교는 로컬에서 공통 3,039장 기준으로 따로 맞춰 계산한다 (옛 실험은 면적 10% 제외가 있어서 평가 사진 목록이 조금 다르다).

## 8. 런타임 반납
결과 ZIP이 Drive에 있는지 확인한 뒤 실행. 학습 중에는 실행하지 말 것.

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()